# Semantic Coherence Attention Module (SCAM)

SCAM performs mask-aware semantic matching between missing and observed feature patches and uses the most similar observed patches to fill missing feature regions.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


## 1. Known and Unknown Patch Partition

Local feature patches are categorized into gap and non-gap groups according to the mask.


In [ ]:
gap_mask = (patch_mask == 1)
nongap_mask = (patch_mask == 0)

gap_patches = patches[..., gap_mask]
nongap_patches = patches[..., nongap_mask]


## 2. Cosine Similarity

Gap and non-gap patches are flattened and normalized before computing cosine similarity.


In [ ]:
D = C * ks * ks
gap_flat = gap_patches.reshape(D, -1)
nongap_flat = nongap_patches.reshape(D, -1)

gap_norm = F.normalize(gap_flat, dim=0)
nongap_norm = F.normalize(nongap_flat, dim=0)
sim = torch.matmul(nongap_norm.t(), gap_norm)


## 3. Top-K Semantic Matching

For each gap patch, the top-K most similar non-gap patches are selected and converted to temperature-controlled softmax weights.


In [ ]:
K = min(self.top_k, sim.size(0))
topk_sim, topk_idx = torch.topk(sim, k=K, dim=0)
weights = F.softmax(topk_sim / self.temperature, dim=0)


## 4. Weighted Patch Fusion

The selected non-gap patches are fused according to their similarity weights.


In [ ]:
sel = nongap_flat[:, topk_idx]

fused = torch.sum(sel * weights.unsqueeze(0), dim=1)

fused = fused.view(C, ks, ks, -1)
